# 4. Settings

Understand configuration, deadlines and qualification policy before changing a device.


## Before you start

Follow [the notebook setup and safety guide](README.md), then select the
project's Python 3.12 kernel. Run cells from top to bottom. Each notebook
is independent. Default execution uses a fictional config and makes no
hardware calls; a skipped operation is **not a successful hardware test**.

For an authorized live session, set `CONFIG` to `ROOT / 'shutters.local.toml'`,
set `TARGET` to the reviewed exact device name, and only then enable the
relevant gate. Keep the source independently protected. Release any GUI,
CLI or other notebook owner first. Restart the kernel after changing setup.


In [ ]:
from pathlib import Path
from dataclasses import asdict
from pprint import pprint
from thorlabs_shutter_control import ShutterHub, load_config

# Works from the repository root or docs/examples. No device access.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'config/shutters.example.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project checkout.')
CONFIG = ROOT / 'config/shutters.example.toml'  # Fictional; offline only.
TARGET = 'example'  # Exact configured name; never select the first device.
RUN_HARDWARE = False

configs = load_config(CONFIG)
selected = next((c for c in configs if c.name == TARGET), None)
if selected is None:
    raise ValueError('TARGET must match an exact configured name.')
print('Selected:', selected.name, selected.serial_number, selected.route.kind)


## 1. Inspect local settings without writing them

Config files describe exact identities and operating permissions. Loading
them does not apply device settings. Application `channel` is a logical
index; the KSC101 protocol channel is always 1. Do not edit the fictional
example into a real policy: copy it to ignored `shutters.local.toml` first.


In [ ]:
pprint(asdict(selected))
print('Read-only:', selected.policy is None)
hub = ShutterHub(configs, connect_timeout=5.0,
                 query_timeout=3.0, cleanup_timeout=5.0)
try:
    print('Host budgets (s):', hub.connect_timeout,
          hub.query_timeout, hub.cleanup_timeout)
finally:
    hub.close()


## 2. Know which settings belong where

| Setting | Purpose / limits |
| --- | --- |
| `name`, `channel`, `role` | Explicit logical-to-physical mapping |
| `serial_number`, `firmware`, `route` | Exact identity and permitted route |
| `connect_timeout` | Whole connection/identity budget; default 5 s |
| `query_timeout` | Whole ordinary operation budget; default 3 s |
| `cleanup_timeout` | Separate cleanup budget; default 5 s |
| Omitted `policy` | Read-only; even OFF requires qualification |
| `policy.evidence`, `shutter_model` | Traceable actual-load qualification |
| `off_optical_state` | Independently demonstrated OFF → closed mapping |
| `min_off_s` | Qualified minimum OFF interval in seconds |
| `continuous_on_allowed` | Actual load must support continuous manual ON |
| `allow_persistent_mode_changes` | Separate permission for a remembered mode change |

Do not guess ratings, set `min_off_s=0` for convenience, or fabricate an
evidence string. A policy is a record of qualification, not a safety interlock.
Finite exposure/duty-limited loads remain unavailable pending an independent
cutoff and qualification. Native calls may exceed requested timeouts;
these budgets are not guaranteed physical closure deadlines.

## 3. Device settings intentionally excluded

The public API does not expose timing/cycle writes, trigger priming,
interlock bypass, EEPROM writes or network reconfiguration. Manual selection
is demonstrated separately in notebook 5; ON/OFF is in notebook 6.
See [setup](../getting-started.md) and [configuration example](../../config/shutters.example.toml).
